In [2]:
from w3_profiling import profiling
from main import w1_main, w2_main, w3_main, benchmark_all, benchmark_dtype, benchmark_numba_imp

## Comparison of Naive python, Numpy and Jit implementations

In [2]:
w1, w2, w03, w3 = benchmark_all()

Week 1: Naive python implementatio 
Execution took: 7.232399 seconds
Execution took: 7.817936 seconds
Execution took: 7.073860 seconds
Median execution time over 3 runs: 7.232576 seconds
Week 2: numpy vectorization
Execution took: 1.060532 seconds
Execution took: 1.045350 seconds
Execution took: 1.062748 seconds
Median execution time over 3 runs: 1.081353 seconds
Week 3: Naive numba
Execution took: 1.284018 seconds
Execution took: 0.605786 seconds
Execution took: 0.634039 seconds
Median execution time over 3 runs: 0.634904 seconds
Weel 3: optimized numba
Execution took: 0.241691 seconds
Execution took: 0.081422 seconds
Execution took: 0.079074 seconds
Median execution time over 3 runs: 0.093119 seconds


As we can see, numpy is around 7 times faster than the naive approach, but after initialisation numba is about 8 times faster than that.

In [4]:
benchmark_numba_imp(n_runs=4) # Use first run as a warmup

full jit approach
Execution took: 0.095891 seconds
Execution took: 0.083767 seconds
Execution took: 0.088183 seconds
Execution took: 0.082480 seconds
Median execution time over 4 runs: 0.096988 seconds
Python loops approach
Execution took: 0.607892 seconds
Execution took: 0.633925 seconds
Execution took: 0.585210 seconds
Execution took: 0.581039 seconds
Median execution time over 4 runs: 0.596822 seconds
Parallel numba
Execution took: 0.024654 seconds
Execution took: 0.025298 seconds
Execution took: 0.024224 seconds
Execution took: 0.024150 seconds
Median execution time over 4 runs: 0.033895 seconds


As we can see the approach of using jit for the loops gives a ~6x speedup and is comparable to numpy. 
As for parallel there's a ~3x increase

## Profiling

In [7]:
profiling('w1_main(1024)', 'w1')
profiling('w2_main(1024)', 'w2')

Execution took: 0.968913 seconds
Thu Feb 26 14:00:39 2026    w1.prof

         1753669 function calls (1753664 primitive calls) in 1.063 seconds

   Ordered by: cumulative time
   List reduced from 168 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      2/1    0.000    0.000    0.969    0.969 {built-in method builtins.exec}
        1    0.099    0.099    0.969    0.969 <string>:1(<module>)
        1    0.000    0.000    0.858    0.858 /Users/astatham/Documents/Development/NumericalComputingCourse/main.py:15(wrapper)
        1    0.000    0.000    0.858    0.858 /Users/astatham/Documents/Development/NumericalComputingCourse/main.py:35(w1_main)
        1    0.631    0.631    0.858    0.858 /Users/astatham/Documents/Development/NumericalComputingCourse/main.py:49(w1_f)
  1742918    0.245    0.000    0.245    0.000 {built-in method builtins.abs}
        2    0.000    0.000    0.099    0.049 /Users/astatham/miniforge3/envs/conda_venv/

## Week 1 line profiler

Line No.      Hits         Time  Per Hit   % Time  Line Contents

==============================================================

    37                                           @timing
    38                                           @profile
    39                                           def w1_main(max_iters: int = 100, 
    40                                                   x_set: tuple = (-2.0, 1.0),    # Changed to tuple + floats
    41                                                   y_set: tuple = (-1.5, 1.5),    # Changed to tuple + floats
    42                                                   win_size: int = 100) -> np.ndarray :
    43                                               """Generate and plot the Mandelbrot set.
    44                                               Args: 
    45                                                   max_iter (int): Maximum number of iterations.
    46                                                   x_set (tuple): X-axis range.
    47                                                   y_set (tuple): Y-axis range.
    48                                                   win_size (int): Number of points in each axis.
    49                                               Returns:
    50                                                   np.ndarray: Mandelbrot set values in a 2D array.
    51                                               """
    52         1          4.0      4.0      0.0      def w1_f(points: np.ndarray[complex], mandelbrot_Set: np.ndarray[int], max_iters: int = 100) -> np.ndarray:
    53                                                       for i, c in enumerate(points):
    54                                                           z = 0
    55                                                           for j in range(max_iters):
    56                                                               z = z**2 + c
    57                                                               # Break if |Z| > 2
    58                                                               if abs(z) > 2:
    59                                                                   mandelbrot_set[i] = j
    60                                                                   break
    61                                                           else:
    62                                                               mandelbrot_set[i] = max_iters
    63                                                       return mandelbrot_set
    64                                           
    65                                           
    66         1          2.0      2.0      0.0      x_set, y_set = [-2, 1], [-1.5, 1.5]
    67         1        182.0    182.0      0.0      width = np.linspace(x_set[0], x_set[1], win_size)
    68         1         50.0     50.0      0.0      height = np.linspace(y_set[0], y_set[1], win_size)
    69         1          0.0      0.0      0.0      points = []
    70      1025        552.0      0.5      0.0      for w in width:
    71   1049600     487661.0      0.5      1.7          for h in height:
    72   1048576    2475414.0      2.4      8.5              points.append(w + h*1j)
    73         1      70138.0  70138.0      0.2      points = np.array(points)
    74                                           
    75                                               # Based on the 100x100 points, compute the mandelbrot set
    76         1        284.0    284.0      0.0      mandelbrot_set = np.zeros(points.shape, dtype=int)
    77                                               # Get n and c
    78         1   26051135.0 2.61e+07     89.6      mandelbrot_set = w1_f(points, mandelbrot_set, max_iters)
    79                                               # Reshape to 2D and plot        
    80         1         46.0     46.0      0.0      mandelbrot_set = np.reshape(mandelbrot_set, (len(width), len(height))).T
    81         1          1.0      1.0      0.0      return mandelbrot_set

## Numpy line profile
```
Function: w2_main at line 82

Line No      Hits         Time  Per Hit   % Time  Line Contents

==============================================================

    82                                           @profile
    83                                           def w2_main(max_iters: int = 100, 
    84                                                   x_set: tuple = (-2.0, 1.0),    # Changed to tuple + floats
    85                                                   y_set: tuple = (-1.5, 1.5),    # Changed to tuple + floats
    86                                                   win_size: int = 100) -> np.ndarray :
    87                                               """Generate and plot the Mandelbrot set.
    88                                               Args: 
    89                                                   max_iter (int): Maximum number of iterations.
    90                                                   x_set (tuple): X-axis range.
    91                                                   y_set (tuple): Y-axis range.
    92                                                   win_size (int): Number of points in each axis.
    93                                               Returns:
    94                                                   np.ndarray: Mandelbrot set values in a 2D array.
    95                                               """
    96         2         28.0     14.0      0.0      @timing
    97         2          2.0      1.0      0.0      def f(C: np.ndarray[complex], Z: np.ndarray, M: np.ndarray, max_iters: int) -> np.ndarray:
    98                                                   for _ in range(max_iters):
    99                                                       # Calculate the mask for every point in Z
   100                                                       mask = np.abs(Z) <= 2
   101                                                       # Calculate the new Z values 
   102                                                       Z[mask] = Z[mask]**2 + C[mask]
   103                                                       # Update M for points that are still within the escape radius
   104                                                       M[mask] += 1
   105                                                   return M
   106                                           
   107         1        139.0    139.0      0.0      width = np.linspace(x_set[0], x_set[1], win_size)
   108         1         44.0     44.0      0.0      height = np.linspace(y_set[0], y_set[1], win_size)
   109         1       8743.0   8743.0      0.7      X, Y = np.meshgrid(width, height)
   110         1      25774.0  25774.0      2.0      C = X + 1j * Y
   111                                               
   112                                               # Based on the 100x100 points, compute the mandelbrot set
   113         1       1479.0   1479.0      0.1      mandelbrot_set = np.zeros(C.shape, dtype=C.dtype)
   114                                           
   115                                               # Initialize Z and M arrays
   116         1       7463.0   7463.0      0.6      Z = np.zeros_like(C, dtype=complex)
   117         1       3458.0   3458.0      0.3      M = np.zeros_like(C, dtype=int) 
   118                                               # Compute the Mandelbrot set using the function f 
   119         1    1265805.0 1.27e+06     96.4      mandelbrot_set = f(C, Z, M, max_iters)
   120         1          1.0      1.0      0.0      return mandelbrot_set
   ```

## Numba 

```
Line No      Hits         Time  Per Hit   % Time  Line Contents

==============================================================
   214                                           @profile
   215                                           def w3_main(max_iters: int = 100, 
   216                                                   x_set: tuple = (-2.0, 1.0),    # Changed to tuple + floats
   217                                                   y_set: tuple = (-1.5, 1.5),    # Changed to tuple + floats
   218                                                   win_size: int = 100,
   219                                                   dtype: np.dtype = np.float64) -> np.ndarray :
   220                                               """Generate and plot the Mandelbrot set.
   221                                               Args: 
   222                                                   max_iter (int): Maximum number of iterations.
   223                                                   x_set (tuple): X-axis range.
   224                                                   y_set (tuple): Y-axis range.
   225                                                   win_size (int): Number of points in each axis.
   226                                               Returns:
   227                                                   np.ndarray: Mandelbrot set values in a 2D array.
   228                                               """
   229         1        155.0    155.0      0.0      width = np.linspace(x_set[0], x_set[1], win_size)
   230         1         43.0     43.0      0.0      height = np.linspace(y_set[0], y_set[1], win_size)
   231         1       8509.0   8509.0      0.4      X, Y = np.meshgrid(width, height)
   232         1      21823.0  21823.0      0.9      C = X + 1j * Y
   233                                               
   234                                               # Based on the 100x100 points, compute the mandelbrot set
   235         1         27.0     27.0      0.0      mandelbrot_set = np.zeros(C.shape, dtype=np.int32)
   236                                               # Compute the Mandelbrot set using the function f 
   237         1    2275248.0 2.28e+06     98.7      mandelbrot_set = w3_f(C, mandelbrot_set, max_iters)
   238         1          1.0      1.0      0.0      return mandelbrot_set
   ```


We can see that abs is called many times in the naive implementation, taking up a large amount of time, this is not the case in numpy

## float type comparison (64/32)

In [3]:
benchmark_dtype(10)

Execution took: 0.882239 seconds
Execution took: 0.084576 seconds
Execution took: 0.083964 seconds
Execution took: 0.082063 seconds
Execution took: 0.084141 seconds
Execution took: 0.082160 seconds
Execution took: 0.081549 seconds
Execution took: 0.084220 seconds
Execution took: 0.081376 seconds
Execution took: 0.083046 seconds
Median execution time over 10 runs: 0.092563 seconds
Execution took: 0.235203 seconds
Execution took: 0.088830 seconds
Execution took: 0.082576 seconds
Execution took: 0.084512 seconds
Execution took: 0.086323 seconds
Execution took: 0.084678 seconds
Execution took: 0.084784 seconds
Execution took: 0.088643 seconds
Execution took: 0.086291 seconds
Execution took: 0.084248 seconds
Median execution time over 10 runs: 0.091267 seconds


We see no considerable speedup between float64 (top) and float32 (bottom)